In [2]:
import pandas as pd

# =========================
# 1. LOAD DATA
# =========================
file_path = "Atlantic_South_Korea.csv"   # make sure file is in same folder
df = pd.read_csv(r"C:\Users\Jackson Danie M D\Downloads\Atlantic_South_Korea.csv")

# =========================
# 2. INITIAL INSPECTION
# =========================
print("\n--- HEAD ---")
print(df.head())

print("\n--- INFO ---")
print(df.info())

print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

# =========================
# 3. DATA TYPE FIXES
# =========================
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')
df['duration_ms'] = pd.to_numeric(df['duration_ms'], errors='coerce')

# =========================
# 4. DROP INVALID ROWS
# =========================
df = df.dropna(subset=['date', 'position', 'popularity', 'duration_ms'])

# =========================
# 5. CREATE UNIQUE SONG IDENTIFIER
# =========================
df['song_artist'] = (
    df['song'].astype(str).str.strip().str.lower() + "_" +
    df['artist'].astype(str).str.strip().str.lower()
)

# =========================
# 6. CONVERT DURATION (ms → minutes)
# =========================
df['duration_min'] = df['duration_ms'] / 60000

# =========================
# 7. REMOVE DUPLICATES
# =========================
df = df.drop_duplicates(subset=['date', 'song_artist'])

# =========================
# 8. VALIDATE 50 SONGS PER DAY
# =========================
daily_counts = df.groupby('date').size()

print("\n--- DAILY ENTRY COUNT ---")
print(daily_counts.value_counts())

# Keep only valid days (exactly 50 entries)
valid_dates = daily_counts[daily_counts == 50].index
df = df[df['date'].isin(valid_dates)]

# =========================
# 9. SORT DATA (VERY IMPORTANT)
# =========================
df = df.sort_values(['song_artist', 'date'])
df = df.reset_index(drop=True)

# =========================
# 10. FINAL CHECK
# =========================
print("\n--- FINAL SHAPE ---")
print(df.shape)

print("\n--- FINAL SUMMARY ---")
print(df.describe())

print("\n--- UNIQUE COUNTS ---")
print(df.nunique())

# =========================
# 11. SAVE CLEAN DATA (IMPORTANT)
# =========================
df.to_csv("cleaned_kpop_data.csv", index=False)

print("\n✅ Data Cleaning Completed Successfully!")


--- HEAD ---
         date  position                             song       artist  \
0  18-05-2024         1                       Like Crazy        Jimin   
1  18-05-2024         2  UNFORGIVEN (feat. Nile Rodgers)  LE SSERAFIM   
2  18-05-2024         3                            Spicy        aespa   
3  18-05-2024         4                             I AM          IVE   
4  18-05-2024         5                        Queencard     (G)I-DLE   

   popularity  duration_ms album_type  total_tracks  is_explicit  \
0          93       212241     single             6        False   
1          87       182148      album            13        False   
2          81       197040     single             6        False   
3          89       183853      album            11        False   
4          71       161240     single             6        False   

                                     album_cover_url  
0  https://i.scdn.co/image/ab67616d0000b2732b4607...  
1  https://i.scdn.co/image/a

C:\Users\Jackson Danie M D\AppData\Local\Temp\ipykernel_20724\1521960803.py:24: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')


date               554
position            50
song               527
artist             194
popularity          79
duration_ms        512
album_type           3
total_tracks        27
is_explicit          2
album_cover_url    394
song_artist        537
duration_min       512
dtype: int64

✅ Data Cleaning Completed Successfully!


In [5]:
import pandas as pd

# =========================
# 1. LOAD CLEANED DATA
# =========================
df = pd.read_csv(r"C:\Users\Jackson Danie M D\Downloads\Atlantic_South_Korea.csv")
df['date'] = pd.to_datetime(df['date'])

# =========================
# 2. STANDARDIZE COLUMNS
# =========================
df.columns = df.columns.str.lower().str.strip()

required_cols = ['date', 'song', 'artist', 'position', 'popularity']
missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# =========================
# 3. DATA TYPE FIXES
# =========================
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')

# Drop bad rows
df = df.dropna(subset=['date', 'position', 'popularity'])

# =========================
# 4. CREATE UNIQUE IDENTIFIER
# =========================
df['song_artist'] = (
    df['song'].astype(str).str.strip().str.lower() + "_" +
    df['artist'].astype(str).str.strip().str.lower()
)

# =========================
# 5. SORT DATA (CRITICAL)
# =========================
df = df.sort_values(['song_artist', 'date']).reset_index(drop=True)

# =========================
# 6. PREVIOUS APPEARANCE
# =========================
df['prev_date'] = df.groupby('song_artist')['date'].shift(1)

# =========================
# 7. GAP CALCULATION
# =========================
df['gap_days'] = (df['date'] - df['prev_date']).dt.days

# =========================
# 8. ENTRY CLASSIFICATION
# =========================
def classify_entry(row):
    if pd.isna(row['prev_date']):
        return 'first_entry'
    elif row['gap_days'] > 1:
        return 're_entry'
    else:
        return 'continuous'

df['entry_type'] = df.apply(classify_entry, axis=1)

# =========================
# 9. EXIT DETECTION
# =========================
df['next_date'] = df.groupby('song_artist')['date'].shift(-1)
df['next_gap'] = (df['next_date'] - df['date']).dt.days

df['is_exit'] = df['next_gap'] > 1

# =========================
# 10. RE-ENTRY COUNT
# =========================
reentry_counts = (
    df[df['entry_type'] == 're_entry']
    .groupby('song_artist')
    .size()
    .reset_index(name='reentry_count')
)

df = df.merge(reentry_counts, on='song_artist', how='left')
df['reentry_count'] = df['reentry_count'].fillna(0)

# =========================
# 11. RE-ENTRY GAP (KPI)
# =========================
df['reentry_gap'] = df['gap_days'].where(df['entry_type'] == 're_entry')

# =========================
# 12. SUMMARY TABLE
# =========================
summary = df.groupby('song_artist').agg(
    first_entry_count=('entry_type', lambda x: (x == 'first_entry').sum()),
    reentry_count=('reentry_count', 'max'),
    avg_reentry_gap_days=('reentry_gap', 'mean')
).reset_index()

# =========================
# 13. SAVE OUTPUT
# =========================
df.to_csv("reentry_analysis_data.csv", index=False)
summary.to_csv("reentry_summary.csv", index=False)

# =========================
# 14. DEBUG OUTPUT
# =========================
print("\n--- DATA SAMPLE ---")
print(df[['song_artist', 'date', 'entry_type', 'gap_days']].head(20))

print("\n--- TOP RE-ENTRY SONGS ---")
print(summary.sort_values('reentry_count', ascending=False).head(10))

print("\n--- ENTRY TYPE DISTRIBUTION ---")
print(df['entry_type'].value_counts())

print("\n✅ STEP 2 COMPLETED SUCCESSFULLY")

C:\Users\Jackson Danie M D\AppData\Local\Temp\ipykernel_20724\4069279185.py:7: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date'] = pd.to_datetime(df['date'])



--- DATA SAMPLE ---
                                    song_artist       date   entry_type  \
0                   1-800-hot-n-fun_le sserafim 2025-09-01  first_entry   
1                   1-800-hot-n-fun_le sserafim 2025-09-02   continuous   
2                   1-800-hot-n-fun_le sserafim 2025-09-03   continuous   
3                   1-800-hot-n-fun_le sserafim 2025-09-04   continuous   
4                   1-800-hot-n-fun_le sserafim 2025-09-05   continuous   
5                   1-800-hot-n-fun_le sserafim 2025-09-06   continuous   
6                   1-800-hot-n-fun_le sserafim 2025-09-07   continuous   
7                   1-800-hot-n-fun_le sserafim 2025-09-08   continuous   
8                      24yb (intro)_yanghongwon 2025-05-26  first_entry   
9                      24yb (intro)_yanghongwon 2025-05-27   continuous   
10                     24yb (intro)_yanghongwon 2025-05-28   continuous   
11  3d (alternate ver.)_jung kook & jack harlow 2024-10-02  first_entry   
12  

In [7]:
import pandas as pd
import numpy as np
# =========================
# 2. SORT (MANDATORY)
# =========================
df = df.sort_values(['song_artist', 'date']).reset_index(drop=True)

# =========================
# 3. RANK CHANGE
# =========================
df['prev_rank'] = df.groupby('song_artist')['position'].shift(1)
df['rank_change'] = -(df['position'] - df['prev_rank'])

# =========================
# 4. POPULARITY CHANGE
# =========================
df['prev_pop'] = df.groupby('song_artist')['popularity'].shift(1)

df['pop_change'] = (
    (df['popularity'] - df['prev_pop']) / df['prev_pop']
)

# Clean infinite values
df['pop_change'] = df['pop_change'].replace([np.inf, -np.inf], 0)

# =========================
# 5. MOMENTUM SCORE
# =========================
df['momentum_score'] = df['rank_change'] * df['pop_change']

# =========================
# 6. FOCUS ON ENTRY EVENTS ONLY
# =========================
entry_df = df[df['entry_type'].isin(['first_entry', 're_entry'])].copy()

# =========================
# 7. PEAK RANK AFTER ENTRY
# =========================
df['rolling_best_rank'] = df.groupby('song_artist')['position'].cummin()

# =========================
# 8. DAYS TO PEAK
# =========================
df['entry_date'] = df.groupby('song_artist')['date'].transform('min')
df['days_since_entry'] = (df['date'] - df['entry_date']).dt.days

# =========================
# 9. RECOVERY SPEED
# =========================
df['rank_improvement'] = df['prev_rank'] - df['position']

df['recovery_speed'] = df['rank_improvement'] / (df['days_since_entry'] + 1)

# =========================
# 10. AGGREGATE PER SONG
# =========================
summary = df.groupby('song_artist').agg(
    avg_momentum=('momentum_score', 'mean'),
    max_momentum=('momentum_score', 'max'),
    avg_recovery_speed=('recovery_speed', 'mean'),
    reentry_count=('reentry_count', 'max')
).reset_index()

# =========================
# 11. FANDOM INTENSITY SCORE
# =========================
summary['fandom_score'] = (
    summary['reentry_count'] * summary['avg_momentum']
) / (summary['avg_recovery_speed'] + 1)

# =========================
# 12. CLEAN RESULTS
# =========================
summary = summary.replace([np.inf, -np.inf], 0)
summary = summary.fillna(0)

# =========================
# 13. SAVE OUTPUT
# =========================
df.to_csv("momentum_analysis_data.csv", index=False)
summary.to_csv("momentum_summary.csv", index=False)

# =========================
# 14. DEBUG OUTPUT
# =========================
print("\n--- MOMENTUM SAMPLE ---")
print(df[['song_artist', 'date', 'momentum_score']].head(20))

print("\n--- TOP MOMENTUM SONGS ---")
print(summary.sort_values('avg_momentum', ascending=False).head(10))

print("\n--- TOP FANDOM INTENSITY ---")
print(summary.sort_values('fandom_score', ascending=False).head(10))

print("\n✅ STEP 3 COMPLETED SUCCESSFULLY")


--- MOMENTUM SAMPLE ---
                                    song_artist       date  momentum_score
0                   1-800-hot-n-fun_le sserafim 2025-09-01             NaN
1                   1-800-hot-n-fun_le sserafim 2025-09-02       -0.125000
2                   1-800-hot-n-fun_le sserafim 2025-09-03       -0.000000
3                   1-800-hot-n-fun_le sserafim 2025-09-04       -0.114286
4                   1-800-hot-n-fun_le sserafim 2025-09-05       -0.180556
5                   1-800-hot-n-fun_le sserafim 2025-09-06       -0.000000
6                   1-800-hot-n-fun_le sserafim 2025-09-07       -0.054795
7                   1-800-hot-n-fun_le sserafim 2025-09-08       -0.000000
8                      24yb (intro)_yanghongwon 2025-05-26             NaN
9                      24yb (intro)_yanghongwon 2025-05-27       -0.210526
10                     24yb (intro)_yanghongwon 2025-05-28       -0.428571
11  3d (alternate ver.)_jung kook & jack harlow 2024-10-02             NaN


In [10]:
import pandas as pd
import numpy as np

# =========================
# 1. LOAD RAW DATA
# =========================
df = pd.read_csv(r"C:\Users\Jackson Danie M D\Downloads\Atlantic_South_Korea.csv")

# =========================
# 2. CLEANING
# =========================
df.columns = df.columns.str.lower().str.strip()

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')
df['duration_ms'] = pd.to_numeric(df['duration_ms'], errors='coerce')

df = df.dropna(subset=['date', 'position', 'popularity'])

# Unique ID
df['song_artist'] = (
    df['song'].astype(str).str.lower().str.strip() + "_" +
    df['artist'].astype(str).str.lower().str.strip()
)

# Duration
df['duration_min'] = df['duration_ms'] / 60000

# =========================
# 3. RE-ENTRY DETECTION
# =========================
df = df.sort_values(['song_artist', 'date'])

df['prev_date'] = df.groupby('song_artist')['date'].shift(1)
df['gap_days'] = (df['date'] - df['prev_date']).dt.days

df['entry_type'] = np.where(
    df['prev_date'].isna(), 'first_entry',
    np.where(df['gap_days'] > 1, 're_entry', 'continuous')
)

# =========================
# 4. MOMENTUM CALCULATION
# =========================
df['prev_rank'] = df.groupby('song_artist')['position'].shift(1)
df['rank_change'] = -(df['position'] - df['prev_rank'])

df['prev_pop'] = df.groupby('song_artist')['popularity'].shift(1)
df['pop_change'] = (df['popularity'] - df['prev_pop']) / df['prev_pop']
df['pop_change'] = df['pop_change'].replace([np.inf, -np.inf], 0)

df['momentum_score'] = df['rank_change'] * df['pop_change']

# Recovery
df['days_since_entry'] = df.groupby('song_artist').cumcount()
df['recovery_speed'] = df['rank_change'] / (df['days_since_entry'] + 1)

# =========================
# 5. FIX OPTIONAL COLUMNS
# =========================
df['album_type'] = df.get('album_type', 'unknown')
df['total_tracks'] = pd.to_numeric(df.get('total_tracks', 0), errors='coerce').fillna(0)
df['is_explicit'] = df.get('is_explicit', False)

df['is_explicit'] = df['is_explicit'].astype(str).str.lower().isin(['true','1'])

# =========================
# 6. ANALYSIS
# =========================

# Album Type
album_analysis = df.groupby('album_type').agg(
    avg_momentum=('momentum_score', 'mean'),
    avg_recovery=('recovery_speed', 'mean'),
    count=('song_artist', 'nunique')
).reset_index()

# Explicit
explicit_analysis = df.groupby('is_explicit').agg(
    avg_momentum=('momentum_score', 'mean'),
    avg_recovery=('recovery_speed', 'mean'),
    count=('song_artist', 'nunique')
).reset_index()

# Album Size
size_analysis = df.groupby(pd.cut(df['total_tracks'], bins=[0,5,10,20,50])).agg(
    avg_momentum=('momentum_score', 'mean'),
    count=('song_artist', 'nunique')
).reset_index()

# Duration
duration_analysis = df.groupby(pd.cut(df['duration_min'], bins=[0,2,3,4,6,10])).agg(
    avg_momentum=('momentum_score', 'mean'),
    count=('song_artist', 'nunique')
).reset_index()

# =========================
# 7. OUTPUT
# =========================
print("\n--- ALBUM TYPE ---")
print(album_analysis)

print("\n--- EXPLICIT ---")
print(explicit_analysis)

print("\n--- ALBUM SIZE ---")
print(size_analysis)

print("\n--- DURATION ---")
print(duration_analysis)

print("\n✅ FULL PIPELINE COMPLETED")


--- ALBUM TYPE ---
    album_type  avg_momentum  avg_recovery  count
0        album     -0.005261     -0.009874    242
1  compilation           NaN           NaN      1
2       single      0.003480      0.002376    321

--- EXPLICIT ---
   is_explicit  avg_momentum  avg_recovery  count
0        False      0.000502      0.004534    439
1         True     -0.000674     -0.064617     98

--- ALBUM SIZE ---
  total_tracks  avg_momentum  count
0       (0, 5]      0.004556    246
1      (5, 10]     -0.002225    169
2     (10, 20]     -0.004921    133
3     (20, 50]     -0.014775     12

--- DURATION ---
  duration_min  avg_momentum  count
0       (0, 2]     -0.003252     12
1       (2, 3]      0.001270    210
2       (3, 4]      0.001135    262
3       (4, 6]     -0.007624     50
4      (6, 10]     -0.026402      3

✅ FULL PIPELINE COMPLETED


C:\Users\Jackson Danie M D\AppData\Local\Temp\ipykernel_20724\1265093671.py:14: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')
C:\Users\Jackson Danie M D\AppData\Local\Temp\ipykernel_20724\1265093671.py:87: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  size_analysis = df.groupby(pd.cut(df['total_tracks'], bins=[0,5,10,20,50])).agg(
C:\Users\Jackson Danie M D\AppData\Local\Temp\ipykernel_20724\1265093671.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence